**Zonal Statistics, Trends & Correlation:**
Computes per-Quartier zonal statistics (mean LST and NDVI) for each timestep. Calculates long-term warming trends using linear regression and the Mann-Kendall test. Analyses the spatial correlation between LST and NDVI across Quartiere and time using Pearson correlation.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import xarray as xr
import rioxarray
import geopandas as gpd
from scipy import stats
import pymannkendall as mk
import matplotlib.pyplot as plt

**1.** Load the processed Cube and the Statistische Quartiere shapefile

In [12]:
PROCESSED = Path("../data/processed/")

ds_cube = xr.open_dataset(PROCESSED / "02_zurich_west_cube.nc")

shapefile_path = ("../data/vector/statistische_Quartiere.gpkg")
zurich_west_kreise  = [3, 4, 5, 9, 10]

gdf_all = gpd.read_file(
    shapefile_path,
    layer="stzh.adm_statistische_quartiere_map")

gdf_zw = gdf_all[gdf_all["knr"].isin(zurich_west_kreise)].to_crs("EPSG:2056").copy()

print(ds_cube)
print(f"\nTime steps: {len(ds_cube.time)}")
print(f"Quartiere:  {len(gdf_zw)}")
print(gdf_zw[["qname", "kname"]].head(10))  # adjust column name to your data




<xarray.Dataset> Size: 7MB
Dimensions:      (time: 14, y: 240, x: 178)
Coordinates:
  * time         (time) datetime64[ns] 112B 1985-07-01 1988-07-01 ... 2024-07-01
  * y            (y) float64 2kB 1.253e+06 1.253e+06 ... 1.246e+06 1.246e+06
  * x            (x) float64 1kB 2.678e+06 2.678e+06 ... 2.683e+06 2.683e+06
    band         int64 8B ...
Data variables:
    spatial_ref  int64 8B ...
    LST          (time, y, x) float32 2MB ...
    LST_Celsius  (time, y, x) float32 2MB ...
    NDVI         (time, y, x) float32 2MB ...

Time steps: 14
Quartiere:  12
            qname     kname
4            Werd   Kreis 4
5        Sihlfeld   Kreis 3
6     Albisrieden   Kreis 9
10          Höngg  Kreis 10
15  Gewerbeschule   Kreis 5
16    Langstrasse   Kreis 4
17    Escher Wyss   Kreis 5
19           Hard   Kreis 4
21      Wipkingen  Kreis 10
28    Friesenberg   Kreis 3


In [8]:
print(ds_cube)

<xarray.Dataset> Size: 7MB
Dimensions:      (time: 14, y: 240, x: 178)
Coordinates:
  * time         (time) datetime64[ns] 112B 1985-07-01 1988-07-01 ... 2024-07-01
  * y            (y) float64 2kB 1.253e+06 1.253e+06 ... 1.246e+06 1.246e+06
  * x            (x) float64 1kB 2.678e+06 2.678e+06 ... 2.683e+06 2.683e+06
    band         int64 8B ...
    spatial_ref  int64 8B 0
Data variables:
    LST          (time, y, x) float32 2MB ...
    LST_Celsius  (time, y, x) float32 2MB ...
    NDVI         (time, y, x) float32 2MB ...


**2.** Compute zonal statisics per Quartier

In [ ]:
ds_cube = ds_cube.rio.write_crs("EPSG:2056")

zonal_records = []

quartier_col = "qname"
kreis_col = "kname"

for _, quartier_row in gdf_zw.iterrows():
    q_name = quartier_row[quartier_col]
    q_kreis = quartier_row[kreis_col]
    q_geom = [quartier_row.geometry]


    try:
        ds_q = ds_cube.rio.clip(q_geom, gdf_zw.crs, drop=True, all_touched=True)
    except Exception:
        print(f"  Skipping {q_name} (no overlap or empty geometry)")
        continue
    

    for t in ds_q.time.values:
        lst_vals  = ds_q["LST"].sel(time=t).values
        ndvi_vals = ds_q["NDVI"].sel(time=t).values
        
        zonal_records.append({
            "quartier":  q_name,
            "kreis":     q_kreis,
            "date":      pd.Timestamp(t),
            "lst_mean":  float(np.nanmean(lst_vals)),
            "ndvi_mean": float(np.nanmean(ndvi_vals)),
            "lst_std":   float(np.nanstd(lst_vals)),
            "ndvi_std":  float(np.nanstd(ndvi_vals)),
            "n_pixels":  int(np.count_nonzero(~np.isnan(lst_vals))),
        })

zonal_df = pd.DataFrame(zonal_records)
zonal_df = zonal_df.sort_values(["quartier", "date"]).reset_index(drop=True)
print(f"Zonal statistics computed: {len(zonal_df)} rows")
print(zonal_df.head(10))

/opt/miniconda3/envs/urban-heat/lib/python3.11/site-packages/pyproj/crs/_cf1x8.py:515: UserWarning: angle from rectified to skew grid parameter lost in conversion to CF
  warnings.warn(
/opt/miniconda3/envs/urban-heat/lib/python3.11/site-packages/pyproj/crs/_cf1x8.py:515: UserWarning: angle from rectified to skew grid parameter lost in conversion to CF
  warnings.warn(
/opt/miniconda3/envs/urban-heat/lib/python3.11/site-packages/pyproj/crs/_cf1x8.py:515: UserWarning: angle from rectified to skew grid parameter lost in conversion to CF
  warnings.warn(
/opt/miniconda3/envs/urban-heat/lib/python3.11/site-packages/pyproj/crs/_cf1x8.py:515: UserWarning: angle from rectified to skew grid parameter lost in conversion to CF
  warnings.warn(
/opt/miniconda3/envs/urban-heat/lib/python3.11/site-packages/pyproj/crs/_cf1x8.py:515: UserWarning: angle from rectified to skew grid parameter lost in conversion to CF
  warnings.warn(
/opt/miniconda3/envs/urban-heat/lib/python3.11/site-packages/pyproj/cr

Zonal statistics computed: 168 rows
      quartier    kreis       date    lst_mean  ndvi_mean   lst_std  ndvi_std  \
0  Albisrieden  Kreis 9 1985-07-01  300.028809   0.642823  5.108625  0.194375   
1  Albisrieden  Kreis 9 1988-07-01  298.401489   0.638293  4.223654  0.189547   
2  Albisrieden  Kreis 9 1991-07-01  300.193054   0.638112  4.781400  0.190961   
3  Albisrieden  Kreis 9 1994-07-01  302.191498   0.652401  4.962721  0.187485   
4  Albisrieden  Kreis 9 1997-07-01  298.212921   0.665159  4.983158  0.193703   
5  Albisrieden  Kreis 9 2000-07-01  302.456238   0.683637  5.056408  0.194286   
6  Albisrieden  Kreis 9 2003-07-01  303.305298   0.650511  4.880919  0.193460   
7  Albisrieden  Kreis 9 2006-07-01  304.781494   0.663095  5.297804  0.196286   
8  Albisrieden  Kreis 9 2009-07-01  302.719116   0.683398  5.154472  0.196141   
9  Albisrieden  Kreis 9 2012-07-01  303.704132   0.678179  5.100496  0.201648   

   n_pixels  
0      5035  
1      5035  
2      5035  
3      5035  
4 

/opt/miniconda3/envs/urban-heat/lib/python3.11/site-packages/pyproj/crs/_cf1x8.py:515: UserWarning: angle from rectified to skew grid parameter lost in conversion to CF
  warnings.warn(
/opt/miniconda3/envs/urban-heat/lib/python3.11/site-packages/pyproj/crs/_cf1x8.py:515: UserWarning: angle from rectified to skew grid parameter lost in conversion to CF
  warnings.warn(
/opt/miniconda3/envs/urban-heat/lib/python3.11/site-packages/pyproj/crs/_cf1x8.py:515: UserWarning: angle from rectified to skew grid parameter lost in conversion to CF
  warnings.warn(


**4.** Calculate long-term warming and vegetation trends per Quartier

In [ ]:
trend_records = []

# Convert timestamps to numeric values for regression (years as float)
zonal_df["year_float"] = zonal_df["date"].dt.year + (zonal_df["date"].dt.month - 1) / 12.0

for q_name, grp in zonal_df.groupby("quartier"):
    grp = grp.dropna(subset=["lst_mean", "ndvi_mean"]).sort_values("date")
    
    if len(grp) < 5:   # need at least 5 observations for a meaningful trend
        continue
    
    x = grp["year_float"].values
    
    slope_lst, intercept_lst, r_lst, p_lst, se_lst = stats.linregress(x, grp["lst_mean"].values)
    mk_lst = mk.original_test(grp["lst_mean"].values)
    
    # --- NDVI Linear Trend ---
    slope_ndvi, intercept_ndvi, r_ndvi, p_ndvi, se_ndvi = stats.linregress(x, grp["ndvi_mean"].values)
    mk_ndvi = mk.original_test(grp["ndvi_mean"].values)
    
    # --- Total change (last minus first image) ---
    lst_change  = float(grp["lst_mean"].iloc[-1]  - grp["lst_mean"].iloc[0])
    ndvi_change = float(grp["ndvi_mean"].iloc[-1] - grp["ndvi_mean"].iloc[0])

    trend_records.append({
        "quartier":          q_name,
        "kreis":             grp["kreis"].iloc[0],
        "n_scenes":          len(grp),
        # LST trend
        "lst_slope_per_yr":  round(slope_lst, 4),
        "lst_r2":            round(r_lst**2, 4),
        "lst_p_linreg":      round(p_lst, 4),
        "lst_mk_trend":      mk_lst.trend,        
        "lst_mk_p":          round(mk_lst.p, 4),
        "lst_total_change":  round(lst_change, 3),
        # NDVI trend
        "ndvi_slope_per_yr": round(slope_ndvi, 4),
        "ndvi_r2":           round(r_ndvi**2, 4),
        "ndvi_p_linreg":     round(p_ndvi, 4),
        "ndvi_mk_trend":     mk_ndvi.trend,
        "ndvi_mk_p":         round(mk_ndvi.p, 4),
        "ndvi_total_change": round(ndvi_change, 4),
    })


trends_df = pd.DataFrame(trend_records)
print(trends_df.sort_values("lst_slope_per_yr", ascending=False).head(20))


         quartier     kreis  n_scenes  lst_slope_per_yr  lst_r2  lst_p_linreg  \
10           Werd   Kreis 4        14            0.1678  0.4317        0.0107   
7           Höngg  Kreis 10        14            0.1558  0.5981        0.0012   
2      Altstetten   Kreis 9        14            0.1486  0.4971        0.0049   
4     Friesenberg   Kreis 3        14            0.1438  0.5518        0.0023   
11      Wipkingen  Kreis 10        14            0.1367  0.5706        0.0018   
9        Sihlfeld   Kreis 3        14            0.1360  0.4590        0.0078   
6            Hard   Kreis 4        14            0.1353  0.4387        0.0099   
1    Alt-Wiedikon   Kreis 3        14            0.1341  0.4220        0.0119   
8     Langstrasse   Kreis 4        14            0.1325  0.4017        0.0149   
0     Albisrieden   Kreis 9        14            0.1300  0.5195        0.0036   
5   Gewerbeschule   Kreis 5        14            0.1167  0.4487        0.0088   
3     Escher Wyss   Kreis 5 

**5. Pearson Correlation between LST and NDVI across Quartiere and time**
    
Does higher NDVI systematically predict lower LST?

In [18]:
from scipy.stats import pearsonr

corr_records = []

for date_val, grp in zonal_df.groupby("date"):
    grp_clean = grp.dropna(subset=["lst_mean", "ndvi_mean"])
    if len(grp_clean) < 5:
        continue
    r, p = pearsonr(grp_clean["lst_mean"], grp_clean["ndvi_mean"])
    corr_records.append({
        "date":    date_val,
        "pearson_r": round(r, 4),
        "p_value":   round(p, 8),
        "n_quartiere": len(grp_clean),
    })

corr_df = pd.DataFrame(corr_records)
print(corr_df.describe())

                                date  pearson_r       p_value  n_quartiere
count                             14  14.000000  1.400000e+01         14.0
mean   2004-12-30 06:51:25.714285696  -0.972586  2.567571e-05         12.0
min              1985-07-01 00:00:00  -0.991500  0.000000e+00         12.0
25%              1995-04-01 00:00:00  -0.983700  1.000000e-08         12.0
50%              2004-12-30 00:00:00  -0.982200  1.000000e-08         12.0
75%              2014-09-30 06:00:00  -0.977925  4.000000e-08         12.0
max              2024-07-01 00:00:00  -0.857800  3.591000e-04         12.0
std                              NaN   0.033363  9.596612e-05          0.0


**6. Merge trend results back onto the GeoDataFrame for mapping**

In [22]:
gdf_trends = gdf_zw.merge(
    trends_df, 
    left_on=quartier_col, 
    right_on="quartier", 
    how="left"
)

# Export as GeoPackage
output_gpkg = PROCESSED / "03_trends_per_quartier.gpkg"
gdf_trends.to_file(output_gpkg, layer="trends", driver="GPKG")
print(f"Geo-enriched trend table saved: {output_gpkg}")

# Export plain CSVs
zonal_df.to_csv(PROCESSED / "03_zonal_statistics.csv", index=False)
trends_df.to_csv(PROCESSED / "03_trends.csv", index=False)
corr_df.to_csv(PROCESSED  / "03_correlations_over_time.csv", index=False)
print("All analysis outputs saved.")

Geo-enriched trend table saved: ../data/processed/03_trends_per_quartier.gpkg
All analysis outputs saved.
